In [1]:
##### resize .mov

input_mov = "/Volumes/MUSIC_PROD/IMG_0683.MOV"

In [2]:
# -----######-----###### FFmpeg VIDEO CONVERTER (Preserve Quality + Audio) -----######-----######
import subprocess
from pathlib import Path

def _video_1105_mov2mp4_GET_export_ffmpeg(input_path, output_path=None, scale_factor=1.0):
    """
    Convert .mov video to .mp4 using FFmpeg, preserving audio and optional scaling.

    Parameters:
    - input_path: str or Path — input .mov video path
    - output_path: str or Path — optional; defaults to same name with '_converted.mp4'
    - scale_factor: float — scaling factor for resolution (1.0 = original size)
    """
    input_path = Path(input_path)
    if not input_path.exists():
        raise FileNotFoundError(f"❌ File not found: {input_path}")
    
    if output_path is None:
        output_path = input_path.with_name(input_path.stem + "_converted.mp4")

    vf_str = f"scale=iw*{scale_factor}:ih*{scale_factor}" if scale_factor != 1.0 else "scale=iw:ih"

    cmd = [
        "ffmpeg", "-y", "-i", str(input_path),
        "-vf", vf_str,
        "-c:v", "libx264", "-crf", "18", "-preset", "slow",
        "-c:a", "aac", "-b:a", "192k",
        str(output_path)
    ]

    print(f"🎬 Converting: {input_path.name} → {output_path.name}")
    subprocess.run(cmd, check=True)
    print(f"✅ Exported with audio to: {output_path}")


In [ ]:
_video_1105_mov2mp4_GET_export_ffmpeg(input_mov, scale_factor=0.65)  # 85% of original size


# extract and embed sound to video 

In [6]:
# -----######-----######  AUDIO EXTRACT & INTERACTIVE RE-EMBED (FFMPEG)  -----######-----######-----
import os
from subprocess import run

def _vid_1105_ffmpeg_GET_extract_wait_embed(input_mp4):
    """
    1. Extracts audio from input_mp4 to WAV using ffmpeg.
    2. Prompts user to edit it externally.
    3. On ENTER, embeds the edited WAV back into the original MP4's video stream.

    Output: "{base}_with_audio.mp4"
    """
    base, _ = os.path.splitext(input_mp4)
    wav_path = f"{base}.wav"
    output_mp4 = f"{base}_with_audio.mp4"

    # 1. Extract
    print(f"🎧 Extracting audio from: {input_mp4}")
    run(["ffmpeg", "-y", "-i", input_mp4, "-vn", "-acodec", "pcm_s16le", wav_path])
    print(f"✅ Audio saved to: {wav_path}")

    # 2. Prompt
    input("🎛️ Edit the WAV file externally if needed. Press ENTER to re-embed...")

    # 3. Re-embed
    print("🎬 Embedding edited audio into new video...")
    run([
        "ffmpeg", "-y",
        "-i", input_mp4,
        "-i", wav_path,
        "-map", "0:v:0",     # video from original
        "-map", "1:a:0",     # audio from wav
        "-c:v", "copy",      # don't re-encode video
        "-c:a", "aac",       # compress audio to AAC
        output_mp4
    ])

    print(f"✅ Done. Final file: {output_mp4}")
    return output_mp4


In [ ]:
_vid_1105_ffmpeg_GET_extract_wait_embed("/Volumes/MUSIC_PROD/IMG_0683_converted.mp4")


ffmpeg version 7.1.1 Copyright (c) 2000-2025 the FFmpeg developers
  built with Apple clang version 16.0.0 (clang-1600.0.26.6)
  configuration: --prefix=/opt/homebrew/Cellar/ffmpeg/7.1.1_1 --enable-shared --enable-pthreads --enable-version3 --cc=clang --host-cflags= --host-ldflags='-Wl,-ld_classic' --enable-ffplay --enable-gnutls --enable-gpl --enable-libaom --enable-libaribb24 --enable-libbluray --enable-libdav1d --enable-libharfbuzz --enable-libjxl --enable-libmp3lame --enable-libopus --enable-librav1e --enable-librist --enable-librubberband --enable-libsnappy --enable-libsrt --enable-libssh --enable-libsvtav1 --enable-libtesseract --enable-libtheora --enable-libvidstab --enable-libvmaf --enable-libvorbis --enable-libvpx --enable-libwebp --enable-libx264 --enable-libx265 --enable-libxml2 --enable-libxvid --enable-lzma --enable-libfontconfig --enable-libfreetype --enable-frei0r --enable-libass --enable-libopencore-amrnb --enable-libopencore-amrwb --enable-libopenjpeg --enable-libspeex

🎧 Extracting audio from: /Volumes/MUSIC_PROD/IMG_0683_converted.mp4
✅ Audio saved to: /Volumes/MUSIC_PROD/IMG_0683_converted.wav


In [6]:
from moviepy.editor import VideoFileClip, AudioFileClip
print("✅ moviepy is working")


ModuleNotFoundError: No module named 'moviepy.editor'